In [32]:
%pip install dotenv

Note: you may need to restart the kernel to use updated packages.


In [33]:
# 라이브러리 가져오기
import requests
import pandas as pd
import os
import dotenv

In [34]:
dotenv.load_dotenv()
my_key = os.getenv('KAKAO_RESTFULL_KEY')

In [35]:
# Kakao Map API 통해 위도, 경도 데이터 가져오는 함수를 정의
def get_geocoding(place):
    try:
        url = f"https://dapi.kakao.com/v2/local/search/address"
        headers = {"Authorization": f"KakaoAK {my_key}"}
        params = {
            'query' : place
        }
        response = requests.get(url, headers=headers, params=params)
        result = response.json()
        return result["documents"][0]["y"], result["documents"][0]["x"]
    except:
        return pd.notna #데이터 결측치(데이터 없음 또는 유효하지 않는 데이터) 표시

In [36]:
lat = []  #위도
lng = []  #경도

# 장소(주소) 리스트
places = ["서울특별시 종로구 세종대로 175", 
          "서울특별시 서초구 서초동 700", 
          "부산광역시 해운대구 해운대해변로 264"]

i=0
for place in places:        # 순환하면서 값 넣기
    i = i + 1
    try:
        print(i, place)
        # get_geocoding 함수의 리턴값 호출하여 geo_location 변수에 저장
        place_lat, place_lon = get_geocoding(place)
        lat.append(place_lat)
        lng.append(place_lon)
        
    except:
        lat.append('')
        lng.append('')

# 데이터프레임으로 변환하기
df = pd.DataFrame({'위도':lat, '경도':lng}, index=places)

df

1 서울특별시 종로구 세종대로 175
2 서울특별시 서초구 서초동 700
3 부산광역시 해운대구 해운대해변로 264


,위도,경도
서울특별시 종로구 세종대로 175,37.5718478584908,126.976168275947
서울특별시 서초구 서초동 700,37.4810862955299,127.015245160054
부산광역시 해운대구 해운대해변로 264,35.1591069824231,129.160283786856


In [37]:
df = pd.read_csv('https://raw.githubusercontent.com/pia222sk20/python/main/data/sample.csv'
                 ,encoding='cp949')
df_simple = df.loc[:,['상호명','도로명주소']]
df_simple.head()

,상호명,도로명주소
0,하나산부인과,경기도 안산시 단원구 달미로 10
1,타워광명내과의원,서울특별시 강남구 언주로30길 39
2,조정현신경외과의원,경기도 시흥시 중심상가로 178
3,한귀원정신과의원,부산광역시 수영구 수영로 688
4,더블유스토어수지점,경기도 용인시 수지구 문정로 32


In [38]:
#위도 경도를 추가 apply
#df_simple['위경도'] = df_simple['도로명주소'].apply(lambda x : get_geocoding(x))
df_simple['도로명주소'].index

RangeIndex(start=0, stop=91335, step=1)

In [39]:
import random
random_index = random.sample(range(91335),50)
random_index[:10]

[79924, 37921, 3441, 63293, 55557, 64005, 83745, 13701, 11263, 46238]

In [40]:
df_simple = df_simple.dropna()  # 데이터 중에서 None 또는 NaN 없는 데이터 row를 삭제

In [41]:
df_simple_50 =  df_simple.iloc[random_index]
results = [ get_geocoding(addr) for addr in df_simple_50['도로명주소'].values]
results[:10]

[('37.6070075809917', '126.921952781175'),
 ('37.6654906885636', '127.041934071403'),
 ('37.2352227508172', '127.063997961367'),
 ('35.8663570566412', '128.72507639879'),
 ('37.3873374351731', '126.860504313196'),
 ('37.7716684832327', '126.775048386559'),
 ('37.4929905549467', '127.014115205707'),
 ('35.1366795471993', '129.099226780319'),
 ('37.5808992195068', '127.037475054197'),
 ('36.3268256231142', '127.428432228959')]

In [42]:
df_simple_50.loc[:,'위도'], df_simple_50.loc[:,'경도'] = zip(*results)

TypeError: 'function' object is not iterable

In [ ]:

df_simple_50.head()

,상호명,도로명주소
70187,해오름치과의원,경기도 광명시 철산로 36
25204,미래산부인과,경상북도 경산시 경안로 180
9914,프리미엄전자담배멀티샵부,인천광역시 부평구 부평대로 67-2
23859,지성치과의원,대전광역시 유성구 도안대로 511-6
1589,한국병원별관,충청북도 청주시 상당구 단재로 106


In [ ]:
#지도
%pip install folium


   ---------------------------------------- 0/3 [xyzservices]
   ------------- -------------------------- 1/3 [branca]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   ---------------------------------------- 3/3 [folium]

Note: you may need to restart the kernel to use updated packages.


In [ ]:
for i in df_simple_50:
    print(i)   

상호명
도로명주소


In [43]:
for idx, row in df_simple_50.iterrows():
    print(idx, row['위도'], row['경도'])
    break

KeyError: '위도'

In [ ]:
import folium

m = folium.Map(location=(35.8425738690814, 127.140054344904))  # 최초지도 로드할때. 중심좌표
for idx, row in df_simple_50.iterrows():
    folium.Marker(
        location=[ row['위도'], row['경도'] ],
        popup=row['상호명'],
        tooltip=row['상호명'][-2:]
    ).add_to(m)    

m.save('map.html')

KeyError: '위도'